# Notebook 04 — Hierarchisches 2-Stufen-Modell (Final)

```
Stufe 1 (grob):   Still  ──vs──  Essen
                                   │
Stufe 2 (fein):          Apfel · Kaugummi · Skyr · Essen (generisch)
```

---

## Ergebnisse der Voruntersuchung

### Movement-Exclusion-Schwelle (k = 3 / 5 / 7 × Median)

| k | Stufe 1 (alle Feat.) | Stufe 2 (alle Feat.) |
|---|---|---|
| 3 | 99.1 % | 88.7 % |
| **5** | **99.1 %** | **89.6 %** |
| 7 | 99.1 % | 88.8 % |

→ **k = 5** liefert die beste Stufe-2-Genauigkeit und wird verwendet.

### Feature-Selektion (RFECV, 5-fold CV)

| | Alle Features | RFE-Selektion | Δ |
|---|---|---|---|
| Stufe 1 | 99.1 % | **100.0 %** (5 Features) | +0.9 % |
| Stufe 2 | 89.6 % | 88.1 % (18 Features) | −1.5 % |

→ **Stufe 1** läuft auf den 5 RFECV-selektierten Features: `stillness_ratio`, `magnitude_max`, `lin_y_mean`, `lin_y_std`, `yaw_mean`  
→ **Stufe 2** verwendet alle 36 Features, bereinigt um Permutation-Importance-negative Features (berechnet unten).

---
## 1. Setup

In [ ]:
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import welch, butter, sosfiltfilt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

# ── Klassen ───────────────────────────────────────────────────────────────────
CLASSES_RAW    = ['Apfel', 'Kaugummi', 'Skyr', 'Still', 'Essen']
CLASSES_COARSE = ['Still', 'Essen']
CLASSES_FINE   = ['Apfel', 'Kaugummi', 'Skyr', 'Essen']
CLASSES_FULL   = ['Still', 'Apfel', 'Kaugummi', 'Skyr', 'Essen']
TO_COARSE      = {'Apfel': 'Essen', 'Kaugummi': 'Essen',
                  'Skyr':  'Essen', 'Still':    'Still', 'Essen': 'Essen'}

COLORS = {'Apfel': '#e15759', 'Kaugummi': '#4e79a7', 'Skyr': '#f28e2b',
          'Still': '#59a14f', 'Essen': '#b07aa1'}

# ── Modell-Parameter ──────────────────────────────────────────────────────────
FS            = 50.0
TRIM_SECS     = 2
WINDOW_SECS   = 25.0
MIN_TAIL_SECS = 20.0
K             = 5          # Movement-Exclusion-Schwelle = 5 × Median

# Stufe-1-Features (RFECV-selektiert, k=5, LOO-Accuracy 100 %)
FEATURES_S1 = ['stillness_ratio', 'magnitude_max', 'lin_y_mean', 'lin_y_std', 'yaw_mean']

---
## 2. Daten laden

`Still_Sprechen_*` beginnt mit `Still_` → wird automatisch **Still** zugeordnet.

In [ ]:
DATA_DIR = Path('../data/raw')
_SKIP    = {'Metadata.csv', 'Annotation.csv'}

sessions = {cls: [] for cls in CLASSES_RAW}
for zf in sorted(DATA_DIR.glob('*.zip')):
    for cls in CLASSES_RAW:
        if zf.name.startswith(cls + '_'):
            sessions[cls].append(zf)
            break

print('Sitzungen pro Klasse:')
for cls in CLASSES_RAW:
    print(f'  {cls:12s}: {len(sessions[cls])}')

In [ ]:
def preprocess(df):
    df = df.copy()
    t  = df['seconds_elapsed']
    df = df[(t >= t.iloc[0] + TRIM_SECS) &
            (t <= t.iloc[-1] - TRIM_SECS)].reset_index(drop=True)
    df['lin_x']     = df['accelerationX']
    df['lin_y']     = df['accelerationY']
    df['lin_z']     = df['accelerationZ']
    df['magnitude'] = np.sqrt(df['lin_x']**2 + df['lin_y']**2 + df['lin_z']**2)
    return df

preprocessed = {cls: [] for cls in CLASSES_RAW}
for cls in CLASSES_RAW:
    for zf in sessions[cls]:
        with zipfile.ZipFile(zf) as z:
            csv_name = next(f for f in z.namelist()
                            if f.endswith('.csv') and f not in _SKIP)
            with z.open(csv_name) as f:
                preprocessed[cls].append(preprocess(pd.read_csv(f)))

print(f'Trim: ±{TRIM_SECS} s  |  Sitzungsdauern:')
for cls in CLASSES_RAW:
    durs = [df['seconds_elapsed'].iloc[-1] - df['seconds_elapsed'].iloc[0]
            for df in preprocessed[cls]]
    print(f'  {cls:12s}: {len(durs):2d} Sitzungen  {min(durs):.0f}–{max(durs):.0f} s')

---
## 3. Segmentierung — 25-Sekunden-Fenster

In [ ]:
def split_windows(df):
    t = df['seconds_elapsed'].values
    t_start, t_end = t[0], t[-1]
    windows = []
    while t_start + MIN_TAIL_SECS <= t_end:
        t_stop = t_start + WINDOW_SECS
        w = df[(t >= t_start) & (t < t_stop)].reset_index(drop=True)
        if len(w) > 1:
            dur = w['seconds_elapsed'].iloc[-1] - w['seconds_elapsed'].iloc[0]
            if dur >= MIN_TAIL_SECS:
                windows.append(w)
        t_start = t_stop
    return windows

raw_windows = {cls: [] for cls in CLASSES_RAW}
for cls in CLASSES_RAW:
    for df in preprocessed[cls]:
        raw_windows[cls].extend(split_windows(df))

print(f'Rohfenster pro Klasse ({WINDOW_SECS:.0f} s):')
for cls in CLASSES_RAW:
    n = len(raw_windows[cls])
    print(f'  {cls:12s}: {n:4d}')
print(f'  {"Gesamt":12s}: {sum(len(v) for v in raw_windows.values()):4d}')

---
## 4. Movement Exclusion  (k = 5 × Median)

In [ ]:
def movement_mask(df, k=K, global_min=0.02, rolling_window=50):
    threshold = max(global_min, k * df['magnitude'].median())
    roll_max  = df['magnitude'].rolling(rolling_window, center=True, min_periods=1).max()
    return roll_max <= threshold

windows = {cls: [] for cls in CLASSES_RAW}
for cls in CLASSES_RAW:
    for df in raw_windows[cls]:
        clean = df[movement_mask(df)].reset_index(drop=True)
        if len(clean) > 50:
            windows[cls].append(clean)

print(f'Fenster nach Movement Exclusion (k={K}):')
for cls in CLASSES_RAW:
    orig = len(raw_windows[cls])
    kept = len(windows[cls])
    print(f'  {cls:12s}: {orig:4d} → {kept:4d}  (−{orig - kept})')
print(f'  {"Gesamt":12s}: {sum(len(v) for v in windows.values()):4d}')

---
## 5. Feature-Extraktion

In [ ]:
def extract_features(df):
    feats = {}
    for col in ['lin_x', 'lin_y', 'lin_z', 'magnitude']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    feats['stillness_ratio'] = (df['magnitude'] < 0.02).mean()
    feats['movement_events'] = int(
        (df['magnitude'] > df['magnitude'].quantile(0.75)).sum())
    for col in ['rotationRateX', 'rotationRateY', 'rotationRateZ']:
        feats[f'{col}_mean'] = df[col].mean()
        feats[f'{col}_std']  = df[col].std()
        feats[f'{col}_max']  = df[col].abs().max()
    for col in ['pitch', 'roll', 'yaw']:
        feats[f'{col}_mean']  = df[col].mean()
        feats[f'{col}_std']   = df[col].std()
        feats[f'{col}_range'] = df[col].max() - df[col].min()
    nperseg = min(256, len(df) // 2)
    freqs, psd = welch(df['magnitude'].values, fs=FS, nperseg=nperseg)
    chew = (freqs >= 0.5) & (freqs <= 4.0)
    cf, cp = freqs[chew], psd[chew]
    feats['total_power']        = float(psd.sum())
    feats['chew_band_power']    = float(cp.sum())
    feats['rhythmicity']        = (feats['chew_band_power'] / feats['total_power']
                                   if feats['total_power'] > 0 else 0.0)
    feats['dominant_chew_freq'] = float(cf[np.argmax(cp)]) if len(cp) > 0 else 0.0
    return feats

rows, y_raw = [], []
for cls in CLASSES_RAW:
    for df in windows[cls]:
        rows.append(extract_features(df))
        y_raw.append(cls)

X_all  = pd.DataFrame(rows)
y_raw  = np.array(y_raw)
y_coarse = np.array([TO_COARSE[c] for c in y_raw])

eat_mask = y_coarse == 'Essen'
X_eat    = X_all[eat_mask].reset_index(drop=True)
y_fine   = y_raw[eat_mask]

print(f'Gesamt-Samples: {len(X_all)}')
print(f'  Still:        {(y_coarse=="Still").sum()}')
print(f'  Essen gesamt: {(y_coarse=="Essen").sum()}')
for cls in ['Apfel', 'Kaugummi', 'Skyr', 'Essen']:
    print(f'    {cls:10s}: {(y_raw==cls).sum()}')
print(f'\nFeatures gesamt: {X_all.shape[1]}')

---
## 6. Stufe-2-Feature-Bereinigung — Permutation Importance

Features mit negativer mittlerer Permutation Importance schaden der Genauigkeit und werden entfernt.

In [ ]:
clf_pi = RandomForestClassifier(n_estimators=200, random_state=42,
                                 class_weight='balanced')
clf_pi.fit(X_eat, y_fine)
pi = permutation_importance(clf_pi, X_eat, y_fine,
                             n_repeats=15, random_state=42, n_jobs=-1)

pi_mean = pd.Series(pi.importances_mean, index=X_eat.columns)
pi_std  = pd.Series(pi.importances_std,  index=X_eat.columns)

harmful_s2  = pi_mean[pi_mean < 0].index.tolist()
FEATURES_S2 = [f for f in X_eat.columns if f not in harmful_s2]

print(f'Stufe 2: {len(FEATURES_S2)}/{X_eat.shape[1]} Features behalten')
print(f'Entfernte Features ({len(harmful_s2)}): {harmful_s2}')

# Visualisierung
order  = pi_mean.sort_values(ascending=True)
colors = ['#c00000' if v < 0 else ('#cccccc' if v < 0.002 else '#4e79a7')
          for v in order.values]

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(range(len(order)), order.values,
        xerr=pi_std[order.index].values,
        color=colors, edgecolor='white', capsize=2, height=0.7)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.9, linestyle='--', alpha=0.5)
ax.set_xlabel('Δ Accuracy bei Permutation')
ax.set_title('Permutation Importance — Stufe 2\n'
             'Rot = entfernt  |  Grau = neutral  |  Blau = behalten',
             fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb04_permutation_importance_s2.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: reports/images/nb04_permutation_importance_s2.png')

---
## 7. LOO-CV — Stufe 1 & Stufe 2

In [ ]:
def loo_clf(X, y):
    clf = RandomForestClassifier(n_estimators=200, random_state=42,
                                  class_weight='balanced')
    yt, yp = [], []
    for tr, te in LeaveOneOut().split(X):
        clf.fit(X.iloc[tr], y[tr])
        yp.append(clf.predict(X.iloc[te])[0])
        yt.append(y[te][0])
    return np.array(yt), np.array(yp)

# Stufe 1 — 5 selektierte Features
yt1, yp1 = loo_clf(X_all[FEATURES_S1], y_coarse)
acc1 = accuracy_score(yt1, yp1)
print(f'Stufe 1 — Still vs. Essen  '
      f'({len(FEATURES_S1)} Features):  {acc1:.1%}  '
      f'({int(acc1*len(yt1))}/{len(yt1)})')
print(classification_report(yt1, yp1, labels=CLASSES_COARSE, zero_division=0))

# Stufe 2 — bereinigte Features
yt2, yp2 = loo_clf(X_eat[FEATURES_S2], y_fine)
acc2 = accuracy_score(yt2, yp2)
print(f'Stufe 2 — Feinklassifikation  '
      f'({len(FEATURES_S2)} Features):  {acc2:.1%}  '
      f'({int(acc2*len(yt2))}/{len(yt2)})')
print(classification_report(yt2, yp2, labels=CLASSES_FINE, zero_division=0))

---
## 8. End-to-End — Stufe 1 → Stufe 2 (LOO-CV)

In [ ]:
yt_e2e, yp_e2e = [], []
n = len(X_all)

for te_idx in range(n):
    tr = np.ones(n, dtype=bool); tr[te_idx] = False

    X_tr_s1 = X_all[FEATURES_S1].iloc[tr]
    X_te_s1 = X_all[FEATURES_S1].iloc[[te_idx]]
    yc_tr   = y_coarse[tr]
    yr_tr   = y_raw[tr]

    m1 = RandomForestClassifier(n_estimators=200, random_state=42,
                                 class_weight='balanced')
    m1.fit(X_tr_s1, yc_tr)
    pred = m1.predict(X_te_s1)[0]

    if pred == 'Essen':
        eat_tr = yc_tr == 'Essen'
        if len(np.unique(yr_tr[eat_tr])) >= 2:
            X_tr_s2 = X_all[FEATURES_S2].iloc[tr][eat_tr]
            X_te_s2 = X_all[FEATURES_S2].iloc[[te_idx]]
            m2 = RandomForestClassifier(n_estimators=200, random_state=42,
                                         class_weight='balanced')
            m2.fit(X_tr_s2, yr_tr[eat_tr])
            pred = m2.predict(X_te_s2)[0]

    yt_e2e.append(y_raw[te_idx])
    yp_e2e.append(pred)

yt_e2e = np.array(yt_e2e)
yp_e2e = np.array(yp_e2e)
acc_e2e = accuracy_score(yt_e2e, yp_e2e)
print(f'End-to-End:  {acc_e2e:.1%}  ({int(acc_e2e*len(yt_e2e))}/{len(yt_e2e)})')
print()
print(classification_report(yt_e2e, yp_e2e, labels=CLASSES_FULL, zero_division=0))

---
## 9. Ergebnisübersicht

In [ ]:
print('╔══════════════════════════════════════════════════════╗')
print('║            Finale Modell-Genauigkeit (LOO-CV)        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Stufe 1  Still vs. Essen    ({len(FEATURES_S1):2d} Features)  {acc1:>7.1%}  ║')
print(f'║  Stufe 2  Feinklassifikation ({len(FEATURES_S2):2d} Features)  {acc2:>7.1%}  ║')
print(f'║  End-to-End                              {acc_e2e:>7.1%}  ║')
print('╚══════════════════════════════════════════════════════╝')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, (yt, yp, acc, labels, cmap, title) in zip(axes, [
    (yt1,    yp1,    acc1,    CLASSES_COARSE, 'Greens',  f'Stufe 1: Still vs. Essen\n({acc1:.1%}, {len(FEATURES_S1)} Features)'),
    (yt2,    yp2,    acc2,    CLASSES_FINE,   'Blues',   f'Stufe 2: Feinklassifikation\n({acc2:.1%}, {len(FEATURES_S2)} Features)'),
    (yt_e2e, yp_e2e, acc_e2e, CLASSES_FULL,   'Purples', f'End-to-End\n({acc_e2e:.1%})'),
]):
    cm = confusion_matrix(yt, yp, labels=labels)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=labels, yticklabels=labels, ax=ax,
                linewidths=0.5, linecolor='white',
                annot_kws={'size': 12, 'weight': 'bold'}, vmin=0)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Vorhergesagt', fontsize=10)
    ax.set_ylabel('Tatsächlich', fontsize=10)

plt.suptitle(f'Hierarchisches Modell  |  k={K}  |  LOO-CV  |  25-s-Fenster',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/images/nb04_confusion_matrices.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: reports/images/nb04_confusion_matrices.png')

---
## 10. Feature-Wichtigkeit — finale Modelle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, (X, y, title) in zip(axes, [
    (X_all[FEATURES_S1], y_coarse, f'Stufe 1 — Still vs. Essen  ({len(FEATURES_S1)} Features)'),
    (X_eat[FEATURES_S2], y_fine,   f'Stufe 2 — Feinklassifikation  (Top 12 von {len(FEATURES_S2)})'),
]):
    clf = RandomForestClassifier(n_estimators=200, random_state=42,
                                  class_weight='balanced')
    clf.fit(X, y)
    imp = (pd.Series(clf.feature_importances_, index=X.columns)
           .sort_values(ascending=False))
    imp.head(12).plot.bar(ax=ax, color='#4e79a7', edgecolor='white')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Mean Decrease in Impurity')
    ax.tick_params(axis='x', rotation=45)

sns.despine()
plt.suptitle('Feature-Wichtigkeiten — finale Modelle', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/images/nb04_feature_importance.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Gespeichert: reports/images/nb04_feature_importance.png')